# Offensive IT-Tester — Exploratory Data Analysis

Dataset: cleaned web-application payloads (`payloads_clean.csv`, 455 rows).
Each step below answers a concrete question that feeds either the **classifier**
design or a **responsible-AI** deliverable (fairness, explainability, risk).

Run this from the project root *or* from `notebooks/` — the path resolves either way.

In [17]:

import pandas as pd
from pathlib import Path
import re
import sys
from itertools import combinations
from difflib import SequenceMatcher

root = Path.cwd().parent
sys.path.insert(0, str(root))

from config.paths import CLEAN

pd.set_option("display.max_colwidth", 80)

# JSONL = one JSON object per line
df = pd.read_json(CLEAN / "payloads_clean.jsonl", lines=True)

print(f"Total payloads: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

Total payloads: 455
Columns: ['id', 'attack_class', 'payload', 'type', 'severity', 'context', 'description', 'example']


,id,attack_class,payload,type,severity,context,description,example
0,sqli-001,sqli,' OR '1'='1,tautology,high,Login form username input,Basic tautology-based SQL injection,SELECT * FROM users WHERE username = '' OR '1'='1' AND password = ''
1,sqli-002,sqli,"' UNION SELECT username, password FROM users--",union,high,Search input field,Union-based SQL injection to extract data,"SELECT name, description FROM products WHERE name = '' UNION SELECT username..."
2,sqli-003,sqli,'; WAITFOR DELAY '0:0:5'--,blind-time,medium,ID parameter in URL,Blind SQL injection with time delay,SELECT * FROM orders WHERE id = ''; WAITFOR DELAY '0:0:5'--
3,sqli-004,sqli,"' AND 1=CONVERT(int,@@version)--",error-based,medium,User ID input,Error-based SQL injection to reveal database version,"SELECT * FROM users WHERE id = '' AND 1=CONVERT(int,@@version)--"
4,sqli-005,sqli,' AND 1=1--,boolean-blind,medium,Search field,Boolean-based blind SQL injection,SELECT * FROM products WHERE name = '' AND 1=1--


## 1. Class distribution (type) 
### how well-armed the agent is per attack class.

In [18]:
class_counts = df["type"].value_counts()
class_pct = (class_counts / len(df) * 100).round(1)

dist = pd.DataFrame({"count": class_counts, "percent": class_pct})
print(dist)

# Imbalance flag: any class under 10% of the arsenal is a coverage risk
weak = dist[dist["percent"] < 10]
if not weak.empty:
    print("\nUnder-armed classes (agent is nearly blind here):")
    print(weak)

                   count  percent
type                             
CSRF                  95     20.9
Command Injection     88     19.3
SSRF                  85     18.7
reflected             54     11.9
stored                46     10.1
blind-time            33      7.3
tautology             17      3.7
boolean-blind         13      2.9
stacked-queries       11      2.4
union                  9      2.0
error-based            4      0.9

Under-armed classes (agent is nearly blind here):
                 count  percent
type                           
blind-time          33      7.3
tautology           17      3.7
boolean-blind       13      2.9
stacked-queries     11      2.4
union                9      2.0
error-based          4      0.9


## 2. Severity distribution

### The shape of what the governance gate must police.

In [19]:
sev_counts = df["severity"].value_counts()
sev_pct = (sev_counts / len(df) * 100).round(1)

print(pd.DataFrame({"count": sev_counts, "percent": sev_pct}))

# How much of the arsenal is high-risk = how hard the gate works
high_risk = df["severity"].str.lower().isin(["high", "critical"]).sum()
print(f"\nHigh/critical payloads the gate must hold or escalate: "
      f"{high_risk} ({high_risk/len(df)*100:.1f}%)")

# Which classes carry the dangerous payloads
print("\nSeverity by attack class:")
print(pd.crosstab(df["type"], df["severity"]))

          count  percent
severity                
high        242     53.2
medium      138     30.3
critical     63     13.8
low          12      2.6

High/critical payloads the gate must hold or escalate: 305 (67.0%)

Severity by attack class:
severity           critical  high  low  medium
type                                          
CSRF                     12    55    4      24
Command Injection        19    40    0      29
SSRF                     21    29    8      27
blind-time                0     0    0      33
boolean-blind             0     0    0      13
error-based               0     2    0       2
reflected                 0    54    0       0
stacked-queries          11     0    0       0
stored                    0    46    0       0
tautology                 0     7    0      10
union                     0     9    0       0


##  3. Context analysis (context) 

where payloads can be applied; the basis for selection.

In [20]:
ctx_counts = df["context"].value_counts()
print("Payloads available per context (injection-point type):")
print(ctx_counts)

# Contexts with very few payloads = injection points the agent can find
# but barely test
thin = ctx_counts[ctx_counts < 5]
if not thin.empty:
    print("\nThinly-covered contexts:")
    print(thin)

Payloads available per context (injection-point type):
context
URL parameter                              55
Malicious webpage                          46
User input field                           43
Search input                               25
User input                                 20
                                           ..
Decode and execute base64 payload           1
Use newline as command separator            1
Obfuscated PowerShell encoded command       1
Nested command substitution with quotes     1
Indirect command execution via env var      1
Name: count, Length: 230, dtype: int64

Thinly-covered contexts:
context
Search field                               3
Execute whoami command                     3
Login input                                2
HTTP header injection                      2
Nested command substitution                2
                                          ..
Decode and execute base64 payload          1
Use newline as command separator           

## 4. Coverage matrix (type × context) 

The single most useful artifact; empty cells are blind spots.

In [21]:
coverage = pd.crosstab(df["type"], df["context"])
print("Coverage matrix (rows = attack class, cols = context):")
print(coverage)

# Explicitly list the combinations the scanner CANNOT meaningfully test
blind_spots = [(t, c) for t in coverage.index for c in coverage.columns
               if coverage.loc[t, c] == 0]
print(f"\nBlind spots (zero payloads): {len(blind_spots)}")
for t, c in blind_spots:
    print(f"  - {t} in {c}")

Coverage matrix (rows = attack class, cols = context):
context            AWS EC2 instance metadata  Access Docker API  \
type                                                              
CSRF                                       0                  0   
Command Injection                          0                  0   
SSRF                                       1                  1   
blind-time                                 0                  0   
boolean-blind                              0                  0   
error-based                                0                  0   
reflected                                  0                  0   
stacked-queries                            0                  0   
stored                                     0                  0   
tautology                                  0                  0   
union                                      0                  0   

context            Access Elasticsearch  Access IMAP mail server  \
type 

## 5. Destructive-payload detection 

### Scan the raw payload text to validate severity labels and build your gate rules.

In [22]:
# Patterns that indicate a payload proves the vuln by causing real harm
destructive_patterns = {
    "sql_destructive":  r"\b(DROP|DELETE|TRUNCATE|ALTER)\s+(TABLE|DATABASE)\b",
    "shell_destructive": r"rm\s+-rf|mkfs|>\s*/dev/|:\(\)\{",
    "ssrf_metadata":    r"169\.254\.169\.254|metadata\.google|localhost|127\.0\.0\.1|0\.0\.0\.0",
    "file_write":       r">\s*/|dd\s+if=|chmod\s+777",
}

def flag_destructive(payload):
    hits = [name for name, pat in destructive_patterns.items()
            if re.search(pat, str(payload), re.IGNORECASE)]
    return hits

df["destructive_flags"] = df["payload"].apply(flag_destructive)
df["is_destructive"] = df["destructive_flags"].apply(len) > 0

destructive = df[df["is_destructive"]]
print(f"Destructive payloads found: {len(destructive)}")
print(destructive[["type", "severity", "payload", "destructive_flags"]])

# Trust check: are destructive payloads actually labelled high/critical?
mislabelled = destructive[
    ~destructive["severity"].str.lower().isin(["high", "critical"])
]
print(f"\nDestructive but NOT labelled high/critical "
      f"(severity labels can't be trusted alone): {len(mislabelled)}")
print(mislabelled[["type", "severity", "payload"]])

Destructive payloads found: 64
                  type  severity  \
5      stacked-queries  critical   
60     stacked-queries  critical   
282               SSRF      high   
284               SSRF  critical   
285               SSRF  critical   
..                 ...       ...   
362               SSRF       low   
363               SSRF       low   
364               SSRF  critical   
365               SSRF  critical   
406  Command Injection    medium   

                                                        payload  \
5                                         '; DROP TABLE users--   
60                                     '; DROP TABLE sessions--   
282                                           http://127.0.0.1/   
284                    http://169.254.169.254/latest/meta-data/   
285         http://metadata.google.internal/computeMetadata/v1/   
..                                                          ...   
362                                  http://127.0.0.1/#fragment   


### 6. Duplicate and near-duplicate detection 

## Redundancy wastes the request budget and pushes toward rate-limit problems.

In [23]:
# Exact duplicates after normalising whitespace and case
def normalize(p):
    return re.sub(r"\s+", " ", str(p).strip().lower())

df["_norm"] = df["payload"].apply(normalize)
exact_dupes = df[df.duplicated("_norm", keep=False)].sort_values("_norm")
print(f"Exact (normalised) duplicate payloads: {df['_norm'].duplicated().sum()}")
print(exact_dupes[["type", "payload"]])

# Near-duplicates: pairs above a similarity threshold (fine for ~400 rows)
THRESHOLD = 0.9
uniques = df.drop_duplicates("_norm")["_norm"].tolist()
near = []
for a, b in combinations(uniques, 2):
    ratio = SequenceMatcher(None, a, b).ratio()
    if ratio >= THRESHOLD:
        near.append((round(ratio, 2), a, b))

print(f"\nNear-duplicate pairs (>= {THRESHOLD} similar): {len(near)}")
for ratio, a, b in sorted(near, reverse=True)[:15]:
    print(f"  {ratio}: {a[:40]!r} <-> {b[:40]!r}")

df = df.drop(columns="_norm")


Exact (normalised) duplicate payloads: 1
     type            payload
282  SSRF  http://127.0.0.1/
352  SSRF  HtTp://127.0.0.1/

Near-duplicate pairs (>= 0.9 similar): 155
  1.0: "<form action='https://target.com/api/ste" <-> "<form action='https://target.com/api/ste"
  0.99: "<form action='https://target.com/upload'" <-> "<form action='https://target.com/api/upl"
  0.99: "' or 1=(select count(*) from all_users w" <-> "' or 1=(select count(*) from all_users w"
  0.99: "' or 1=(select count(*) from all_users w" <-> "' or 1=(select count(*) from all_users w"
  0.99: "' or 1=(select count(*) from all_users w" <-> "' or 1=(select count(*) from all_users w"
  0.99: "' or 1=(select count(*) from all_users w" <-> "' or 1=(select count(*) from all_users w"
  0.99: "' or 1=(select count(*) from all_users w" <-> "' or 1=(select count(*) from all_users w"
  0.99: "' or 1=(select count(*) from all_users w" <-> "' or 1=(select count(*) from all_users w"
  0.98: "<meta http-equiv='refresh' content='

## 7. Data-quality / missing-metadata check  
### Missing context or severity now breaks selection and gating, not just cosmetics.

In [24]:
# Missing or empty values per field
missing = df.replace(r"^\s*$", pd.NA, regex=True).isna().sum()
print("Missing values per field:")
print(missing)

# Rows the system cannot place (no context) or cannot police (no severity)
unusable = df[df["context"].isna() | df["severity"].isna()]
print(f"\nPayloads that break selection/gating (missing context or severity): "
      f"{len(unusable)}")
print(unusable[["type", "context", "severity", "payload"]])

Missing values per field:
id                     0
attack_class           0
payload                0
type                   0
severity               0
context                0
description            0
example              248
destructive_flags      0
is_destructive         0
dtype: int64

Payloads that break selection/gating (missing context or severity): 0
Empty DataFrame
Columns: [type, context, severity, payload]
Index: []
